# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mansi-cs/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis:
One row represents one content item (page) for a specific client, measured using search and engagement performance signals.

Time Window:
I will use a mid-panel month (March 2026) for development and verification. This avoids using the final month as a development window and follows the warehouse guidance.



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature fields:
- impressions_90d
- clicks_90d
- ctr
- avg_position
- content_age_days

Label:
- is_declining_label

Context fields:
- client_hash_id
- content_hash_id
- report_date

Excluded fields:
- trend_direction (used to create the label, causes leakage)
- trend_pct (used to create the label, causes leakage)
- IDs as model features (client_hash_id, content_hash_id) because they identify records rather than describe content performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query 1 verified that the March 2026 partition contains 9,841,378 rows covering the period from 2026-03-01 to 2026-03-31.

Query 2 verified the grain. No duplicate combinations of report_date, client_hash_id and content_hash_id were found, confirming that one row represents one content item for one client on one day.

Query 3 verified data availability. Out of 9,841,378 rows, only 413,966 have GA4 data available. Therefore I will filter on ga4_data_available IS TRUE when using GA4-based features.

In [ ]:
from google.colab import userdata
import duckdb

HFTOKEN = userdata.get('HFTOKEN')

con = duckdb.connect()

con.sql(f"""
CREATE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HFTOKEN}'
);
""")


┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [ ]:
# query 1

con.sql("""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

,rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [ ]:
# query 2
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,cnt


In [ ]:
con.sql("""
SELECT
COUNT(*) AS total_rows,
SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS available_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

,total_rows,available_rows
0,9841378,413966.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset cannot prove causation. It can only show relationships between content characteristics and performance. History depth differs across clients, so some clients have much more data than others. Many rows do not have GA4 data available, which limits the use of engagement-based features. Results from this slice may not generalize to all clients or future periods.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.